## 1. Installation

In [1]:
!pip install -q -U "transformers>=4.51" accelerate "bitsandbytes>=0.46.1" peft
print("✅ Installation terminée. Maintenant : Exécution → Redémarrer la session")

✅ Installation terminée. Maintenant : Exécution → Redémarrer la session


## 1b. Vérification

In [2]:
import torch, transformers, bitsandbytes, peft
print("transformers :", transformers.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("peft         :", peft.__version__)
print("GPU          :", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")
assert torch.cuda.is_available(), "⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4"

transformers : 5.23.0
bitsandbytes : 0.50.2
peft         : 0.22.0
GPU          : True - Tesla T4


## 2. Connexion à Hugging Face
Collez votre token (huggingface.co → Settings → Access Tokens → type *Read*).
Utile pour les modèles à accès restreint et pour télécharger plus vite.

In [3]:
from huggingface_hub import login
try:
    login()
except Exception as e:
    print("Connexion ignorée :", e)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## 3. Google Drive : dossier de sauvegarde


In [4]:
import os
import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/SmartCampus_LLM"
os.makedirs(DOSSIER, exist_ok=True)
FICHIER_RESULTATS = os.path.join(DOSSIER, "resultats.csv")

if os.path.exists(FICHIER_RESULTATS):
    _df = pd.read_csv(FICHIER_RESULTATS, dtype={"id": str}).fillna("")
    resultats = _df.to_dict("records")
    print(f"♻️ Reprise : {len(resultats)} réponses déjà enregistrées")
    print(_df.groupby(["modele", "mode"]).size())
else:
    resultats = []
    print("Nouveau départ : aucun résultat enregistré")

def sauvegarder():
    pd.DataFrame(resultats).to_csv(FICHIER_RESULTATS, index=False)

Mounted at /content/drive
Nouveau départ : aucun résultat enregistré


## 4. Les données officielles des formations


In [5]:
def lire(chemin):
    if chemin.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(chemin, dtype=str).fillna("")
    return pd.read_csv(chemin, dtype=str, encoding="utf-8").fillna("")

existant = [f for f in ("formations.xlsx", "formations.csv") if os.path.exists(os.path.join(DOSSIER, f))]
if existant:
    formations = lire(os.path.join(DOSSIER, existant[0]))
    print("Fichier trouvé dans le Drive :", existant[0])
else:
    envoye = files.upload()
    nom = list(envoye.keys())[0]
    extension = ".xlsx" if nom.lower().endswith((".xlsx", ".xls")) else ".csv"
    with open(os.path.join(DOSSIER, "formations" + extension), "wb") as f:
        f.write(envoye[nom])
    formations = lire(nom)
    print("Copie enregistrée dans le Drive : formations" + extension)

print(len(formations), "formations chargées")
formations[["ecole", "nom", "frais_national", "frais_international", "source_url"]]

Saving Classeur1.xlsx to Classeur1.xlsx
Copie enregistrée dans le Drive : formations.xlsx
6 formations chargées


,ecole,nom,frais_national,frais_international,source_url
0,IIT,Génie Civil,556DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-civil/
1,IIT,Génie Mécanique,611DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-mecani...
2,IIT,Génie Informatique,778DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-inform...
3,IIT,Génie Industriel,611DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-indust...
4,IIT,Génie des procédés,611DT/9 mois*,234€/9 mois*,https://iit.tn/genie-des-procedes/
5,ISB,License in Business Administration (Licence en...,,,https://isb.tn/program-lba/


## 5. Les deux consignes données au modèle (avec et sans données)

In [6]:
CHAMPS = [
    ("type", "Type"), ("duree_annees", "Durée (années)"), ("conditions_acces", "Conditions d'accès"),
    ("debouches", "Débouchés"), ("description", "Description"),
    ("frais_national", "Frais (étudiants tunisiens)"), ("frais_international", "Frais (étudiants internationaux)"),
    ("source_url", "Source"),
]

def bloc_formation(ligne):
    texte = f"### {ligne['ecole']} — {ligne['nom']}\n"
    for col, label in CHAMPS:
        valeur = str(ligne.get(col, "")).strip()
        texte += f"- {label} : {valeur if valeur else 'non communiqué'}\n"
    return texte

CONTEXTE = "\n".join(bloc_formation(l) for _, l in formations.iterrows())

REGLES_LANGUE = ("Réponds dans la langue et l'écriture de l'étudiant : français, tounsi en lettres arabes, "
                 "ou Arabizi (tounsi en lettres latines avec 3, 7, 9, 5, 8, 2). Sois clair, court et bienveillant.")

SYSTEM_CONTEXTE = (
    "Tu es SmartCampus AI, l'assistant d'orientation de l'IIT et de l'ISB (Sfax, Tunisie).\n"
    "1. Réponds UNIQUEMENT à partir des informations officielles ci-dessous.\n"
    "2. Si une information est « non communiqué » ou absente, dis-le clairement et propose de contacter "
    "l'établissement. N'invente jamais un chiffre.\n"
    "3. " + REGLES_LANGUE + "\n\n=== INFORMATIONS OFFICIELLES ===\n" + CONTEXTE
)

SYSTEM_SANS_CONTEXTE = "Tu es SmartCampus AI, un assistant pour les étudiants tunisiens. " + REGLES_LANGUE

SYSTEMES = {"contexte": SYSTEM_CONTEXTE, "sans_contexte": SYSTEM_SANS_CONTEXTE}
print("Contexte : environ", len(SYSTEM_CONTEXTE) // 4, "tokens")

Contexte : environ 2266 tokens


## 6. Les questions de test
- **mode `contexte`** (C1 à C28) : questions sur les formations, le modèle reçoit les données officielles.
- **mode `sans_contexte`** (D1 à D10) : compréhension du **dialecte** seul, sans données.
- `mots_attendus` : le **corrigé**, jamais montré au modèle (au moins un mot doit apparaître, `|` = OU).
- `piege = oui` : l'information n'existe pas ; une bonne réponse dit qu'elle ne sait pas.

In [7]:
Q_CONTEXTE = [
    # (langue, question, mots_attendus, piege)   — 28 questions, 5 langues, 1 piège par langue
    # --- Français (5) ---
    ("Français", "Quels sont les frais du Génie Informatique à l'IIT ?", "778", "non"),
    ("Français", "Quelles sont les conditions d'accès au Génie Mécanique ?", "préparatoire|prépa|MP|licence", "non"),
    ("Français", "Combien d'années dure la licence Business Administration de l'ISB ?", "3", "non"),
    ("Français", "Quels sont les frais d'inscription en plus des frais de scolarité à l'IIT ?", "800", "non"),
    ("Français", "Quels sont les frais de la licence Business Administration à l'ISB ?", "", "oui"),
    # --- Anglais (5) ---
    ("Anglais",  "What career opportunities are there after Industrial Engineering at IIT?", "logistic|quality|supply|production|consult", "non"),
    ("Anglais",  "Is the Business Administration bachelor at ISB taught in English?", "yes|english", "non"),
    ("Anglais",  "Which specializations are offered in Computer Engineering at IIT?", "GLID|ARSI|SDIA", "non"),
    ("Anglais",  "Which accreditation does the IIT Computer Engineering program have?", "ASIIN|Euro-Inf", "non"),
    ("Anglais",  "Does IIT offer scholarships for engineering students?", "", "oui"),
    # --- Tounsi en lettres arabes (5) ---
    ("Tounsi",   "قداش معاليم الهندسة المدنية في IIT ؟", "556|600", "non"),
    ("Tounsi",   "شنوة الخدمة اللي نجم نخدمها بعد الهندسة الصناعية ؟", "logisti|qualit|supply|لوجست|جودة|إنتاج", "non"),
    ("Tounsi",   "شنية الشروط باش نقرا هندسة الإعلامية ؟", "préparatoire|prépa|licence|إعداد|إجازة|ليسانس", "non"),
    ("Tounsi",   "قداش تزيد المعاليم بعد العام الأول ؟", "5 %|5%|10%", "non"),
    ("Tounsi",   "الهندسة الميكانيكية تتقرا بالفرنساوي ولا بالأنقليزي ؟", "", "oui"),
    # --- Arabizi (9) ---
    ("Arabizi",  "9adeh soum genie info fel IIT ?", "778", "non"),
    ("Arabizi",  "chnouma les parcours elli fama fi genie informatique ?", "GLID|ARSI|SDIA", "non"),
    ("Arabizi",  "3aslema, billehi n7eb nes2lek 3al genie info, chnouma el conditions bech nod5ol ?", "préparatoire|prépa|licence", "non"),
    ("Arabizi",  "salam, ena 3andi bac math w n7eb na9ra fel IIT, chnouma el 5ayarat elli 3andi ?", "prépa|préparatoire|licence", "non"),

    ("Arabizi",  "n7eb n5dem fel cybersécurité, ama parcours ynasbni fi genie info ?", "ARSI", "non"),
    ("Arabizi",  "famma frais d'inscription zeyda walla le ?", "800", "non"),
    ("Arabizi",  "9adeh soum genie electrique fel IIT ?", "", "oui"),
    # --- Mélange de langues (4) ---
    ("Mélange",  "Bonjour, nheb n3ref les débouchés mta3 génie civil", "structure|construction|infrastructure|bureau", "non"),
    ("Mélange",  "Hi, n7eb na9ra business in English, famma programme fel ISB ?", "Business Administration|English|anglais", "non"),
    ("Mélange",  "Salut, genie des procédés yod5el fih el pétrole walla le ?", "pétrol|petrol|énergie|energie", "non"),
    ("Mélange",  "Hello, 9addech men 3am yodhher genie civil ?", "", "oui"),
]

Q_SANS_CONTEXTE = [
    ("Arabizi",  "chnowa ma3neha 'barcha' bel fransa ?", "beaucoup", "non"),
    ("Arabizi",  "traduis en français : ena mech fehem chay", "comprends", "non"),
    ("Tounsi",   "شنوة معناها 'يعيشك' بالفرنسية ؟", "merci", "non"),
    ("Arabizi",  "kifech nqoul 'je suis fatigué' bel tounsi ?", "ta3ban|t3ban|تعبان|3yit|عييت", "non"),
    ("Arabizi",  "chnowa ma3neha 'taw' bel fransa ?", "maintenant|tout de suite|now", "non"),
    ("Tounsi",   "ترجم للفرنسية : نحب نقرا في الجامعة", "étudier|université|universite", "non"),
    ("Arabizi",  "9oulli bel arabizi : comment tu t'appelles ?", "esmek|ismek|chesmek|chnowa esmek|ism", "non"),
    ("Arabizi",  "3lech fel arabizi nekteb '3' w '7' ?", "ع|ح|lettre|letter|7arf|حرف", "non"),
    ("Mélange",  "rani stressé barcha 3al bac, 3andek nasi7a ?", "révis|organis|repos|sommeil|respir|plan|نوم", "non"),
    ("Arabizi",  "jawebni bel tounsi : chnowa a7sen wa9t bech na9ra fel nhar ?", "sbe7|sba7|matin|lil|صباح|ليل", "non"),
]

lignes = [(f"C{i}", "contexte", *q) for i, q in enumerate(Q_CONTEXTE, 1)] + \
         [(f"D{i}", "sans_contexte", *q) for i, q in enumerate(Q_SANS_CONTEXTE, 1)]
df_questions = pd.DataFrame(lignes, columns=["id", "mode", "langue", "question", "mots_attendus", "piege"])
print(df_questions["mode"].value_counts())
df_questions.head()

mode
contexte         26
sans_contexte    10
Name: count, dtype: int64


,id,mode,langue,question,mots_attendus,piege
0,C1,contexte,Français,Quels sont les frais du Génie Informatique à l...,778,non
1,C2,contexte,Français,Quelles sont les conditions d'accès au Génie M...,préparatoire|prépa|MP|licence,non
2,C3,contexte,Français,Combien d'années dure la licence Business Admi...,3,non
3,C4,contexte,Français,Quels sont les frais d'inscription en plus des...,800,non
4,C5,contexte,Français,Quels sont les frais de la licence Business Ad...,,oui


## 7. Les modèles à comparer


In [8]:
MODELES = [
    "Qwen/Qwen2.5-7B-Instruct",              # multilingue (dont arabe et français)
    "Qwen/Qwen3-8B",                         # modèle de base
    "Artess99/derja-qwen3-8b-derja-v2",      # LoRA tounsi posé sur Qwen3-8B (page non documentée)
    # "Qwen/Qwen3-4B",                       # plus petit et rapide
    # "humain-ai/ALLaM-7B-Instruct-preview", # spécialisé arabe
    # "CohereLabs/aya-expanse-8b",           # accepter les conditions sur HF + token
    # "silma-ai/SILMA-9B-Instruct-v1.0",     # 18,5 Go : trop lourd pour Colab gratuit
]

MAX_NEW_TOKENS = 300
TEMPERATURE = 0.3

In [ ]:
### Comment un adaptateur LoRA est chargé

#Certains « modèles » de la liste ne sont pas des modèles complets, mais seulement des **adaptateurs LoRA**.
#Par exemple, `Artess99/derja-qwen3-8b-derja-v2` ne contient qu'un petit fichier d'environ 99 Mo
#(`adapter_model.safetensors`), entraîné par Artess99 pour le dialecte tunisien.

#Grâce à la bibliothèque **`peft`** (installée en cellule 1), la fonction `charger()` assemble tout automatiquement :

#1. elle lit `adapter_config.json` et y trouve le modèle de base : `Qwen/Qwen3-8B` ;
#2. elle télécharge et charge **Qwen3-8B** (environ 16 Go, compressé ici en 4 bits) ;
#3. elle **ajoute l'adaptateur LoRA** par-dessus : Qwen3-8B + adaptateur = modèle spécialisé.""

#Le code est le même pour un modèle complet et pour un adaptateur : seul le nom change.
#On compare ainsi **Qwen3-8B seul** (avant) et **Qwen3-8B + LoRA** (après).

#⚠️ Cet adaptateur a été **entraîné par Artess99**, pas par nous. Notre propre fine-tuning LoRA
#se chargera exactement de la même façon.

## 8. Fonctions (chargement, génération, score)

In [9]:
import gc, time, re, shutil, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

def charger(nom):
    tok = AutoTokenizer.from_pretrained(nom)
    mod = AutoModelForCausalLM.from_pretrained(nom, quantization_config=bnb, device_map="auto")
    mod.eval()
    return tok, mod

def construire_prompt(tok, question, systeme):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    for extra in (dict(enable_thinking=False), dict()):   # enable_thinking=False : Qwen3 sans « réflexion »
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **extra)
        except Exception:
            pass
    fusion = [{"role": "user", "content": systeme + "\n\nQuestion de l'étudiant : " + question}]
    return tok.apply_chat_template(fusion, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generer(tok, mod, question, systeme):
    entrees = tok(construire_prompt(tok, question, systeme), return_tensors="pt").to(mod.device)
    debut = time.time()
    sorties = mod.generate(**entrees, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                           top_p=0.9, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    texte = tok.decode(sorties[0][entrees["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", texte, flags=re.S).strip(), time.time() - debut

def score_auto(reponse, mots_attendus, piege):
    rep = reponse.lower()
    if piege == "oui":   # 1 si le modèle n'invente pas de prix ou de durée
        invente = bool(re.search(r"\d{2,}\s*(dt|dinar|€|eur|دينار)|\d+\s*(ans\b|an\b|années|سنوات|سنين)", rep))
        return 0 if invente else 1
    mots = [m.strip().lower() for m in str(mots_attendus).split("|") if m.strip()]
    return 1 if any(m in rep for m in mots) else 0

def liberer(mod):
    del mod
    gc.collect()
    torch.cuda.empty_cache()

## 9. Lancer les tests (sauvegarde après chaque réponse)


In [10]:
for nom in MODELES:
    deja = {(r["modele"], str(r["id"])) for r in resultats}
    a_faire = df_questions[[(nom, i) not in deja for i in df_questions["id"]]]
    if a_faire.empty:
        print("✔ déjà terminé :", nom)
        continue

    print("\n" + "=" * 70 + f"\n{nom} : {len(a_faire)} questions à faire")
    try:
        tok, mod = charger(nom)
    except Exception as e:
        print("❌ Impossible de charger ce modèle :", e)
        continue

    for _, q in a_faire.iterrows():
        try:
            reponse, duree = generer(tok, mod, q["question"], SYSTEMES[q["mode"]])
        except Exception as e:
            reponse, duree = f"ERREUR : {e}", 0.0
        s = score_auto(reponse, q["mots_attendus"], q["piege"])
        resultats.append({"modele": nom, "id": q["id"], "mode": q["mode"], "langue": q["langue"],
                          "question": q["question"], "piege": q["piege"], "reponse": reponse,
                          "temps_s": round(duree, 1), "score_auto": s})
        sauvegarder()
        print(f"[{q['id']}] ({q['langue']}) {q['question']}\n→ {reponse[:250]}\n   score={s}  temps={duree:.1f}s\n")

    liberer(mod)
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

print("✅ Terminé. Résultats dans :", FICHIER_RESULTATS)


Qwen/Qwen2.5-7B-Instruct : 36 questions à faire


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (29 outputs left out)

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 87.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

[C1] (Français) Quels sont les frais du Génie Informatique à l'IIT ?
→ Prix par mois (étudiants tunisiens) : 778 DT
Prix par mois (étudiants internationaux) : 234 €
   score=1  temps=12.0s

[C2] (Français) Quelles sont les conditions d'accès au Génie Mécanique ?
→ Diplôme de préparatoire (MP/PC/PT), ou licence dans l'une des spécialités suivantes : maintenance industrielle, électromécanique, conception et fabrication mécaniques, conception et production intégrées, construction mécanique et fabrication, industr
   score=1  temps=25.7s

[C3] (Français) Combien d'années dure la licence Business Administration de l'ISB ?
→ 3 ans
   score=1  temps=5.5s

[C4] (Français) Quels sont les frais d'inscription en plus des frais de scolarité à l'IIT ?
→ Le droit d'inscription est de 800 DT pour les Tunisiens et de 800 € pour les internationaux.
   score=1  temps=10.6s

[C5] (Français) Quels sont les frais de la licence Business Administration à l'ISB ?
→ Les frais de la licence Business Administrat

## 10. Résumé
Fonctionne aussi **sans GPU**, à partir du fichier du Drive : dans une session sans GPU, exécutez seulement
les sections **3**, **10** et **11**.

In [11]:
df = pd.read_csv(FICHIER_RESULTATS, dtype={"id": str})
df["score_auto"] = pd.to_numeric(df["score_auto"], errors="coerce")

resume = df.pivot_table(index="modele", columns="mode", values="score_auto", aggfunc="mean")
resume = (resume * 100).round(0)
resume["temps_moyen_s"] = df.groupby("modele")["temps_s"].mean().round(1)
pieges = df[df["piege"] == "oui"].groupby("modele")["score_auto"].mean()
resume["pieges_evites_%"] = (pieges * 100).round(0)

par_langue = (df.pivot_table(index="modele", columns="langue", values="score_auto", aggfunc="mean") * 100).round(0)

print("Scores en % (contexte = SA-33, sans_contexte = SA-34)")
display(resume)
print("Scores en % par langue")
display(par_langue)

Scores en % (contexte = SA-33, sans_contexte = SA-34)


mode,contexte,sans_contexte,temps_moyen_s,pieges_evites_%
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,88.0,10.0,13.9,100.0
Qwen/Qwen2.5-7B-Instruct,88.0,10.0,11.8,100.0
Qwen/Qwen3-8B,77.0,20.0,20.8,100.0


Scores en % par langue


langue,Anglais,Arabizi,Français,Mélange,Tounsi
modele,,,,,
Artess99/derja-qwen3-8b-derja-v2,100.0,29.0,100.0,80.0,86.0
Qwen/Qwen2.5-7B-Instruct,80.0,36.0,100.0,80.0,86.0
Qwen/Qwen3-8B,100.0,43.0,100.0,60.0,43.0


## 11. Export Excel

In [12]:
notation = df.copy()
for col in ["note_comprehension_1a5", "note_bonne_langue_1a5", "note_fidelite_donnees_1a5", "note_clarte_1a5", "commentaire"]:
    notation[col] = ""

FICHIER_XLSX = os.path.join(DOSSIER, "comparaison_llm_smartcampus.xlsx")
with pd.ExcelWriter(FICHIER_XLSX, engine="openpyxl") as w:
    notation.to_excel(w, sheet_name="reponses", index=False)
    resume.to_excel(w, sheet_name="resume")
    par_langue.to_excel(w, sheet_name="par_langue")
    if "df_questions" in globals():
        df_questions.to_excel(w, sheet_name="questions", index=False)

files.download(FICHIER_XLSX)
print("✅ Excel enregistré dans le Drive et téléchargé :", FICHIER_XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Excel enregistré dans le Drive et téléchargé : /content/drive/MyDrive/SmartCampus_LLM/comparaison_llm_smartcampus.xlsx


## 12. Analyse de la tokenisation (Arabizi vs français vs tounsi)

In [1]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
for phrase in ["combien coûte le génie informatique ?",
               "9addech soum genie info ?",
               "قداش معاليم الهندسة الإعلامية ؟"]:
    morceaux = tok.tokenize(phrase)
    print(len(morceaux), "tokens :", morceaux)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

11 tokens : ['comb', 'ien', 'Ġco', 'Ã»', 'te', 'Ġle', 'ĠgÃ©n', 'ie', 'Ġinformat', 'ique', 'Ġ?']
9 tokens : ['9', 'ad', 'de', 'ch', 'Ġsou', 'm', 'Ġgenie', 'Ġinfo', 'Ġ?']
11 tokens : ['ÙĤØ¯', 'Ø§Ø´', 'ĠÙħØ¹', 'Ø§ÙĦ', 'ÙĬÙħ', 'ĠØ§ÙĦÙĩÙĨØ¯', 'Ø³Ø©', 'ĠØ§ÙĦØ¥Ø¹ÙĦØ§Ùħ', 'ÙĬØ©', 'ĠØ', 'Ł']


In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
phrases = {
    "Français": "combien coûtent les études en génie informatique ?",
    "Anglais":  "how much does computer engineering cost ?",
    "Arabizi":  "9addech yeklef 9raya fi genie informatique ?",
    "Tounsi":   "قداش تكلف القراية في الهندسة الإعلامية ؟",
}
for langue, phrase in phrases.items():
    morceaux = tok.tokenize(phrase)
    mots = len(phrase.split())
    print(f"{langue:9s} {len(morceaux):3d} tokens / {mots} mots = {len(morceaux)/mots:.1f} tokens par mot")

Français   13 tokens / 8 mots = 1.6 tokens par mot
Anglais     7 tokens / 7 mots = 1.0 tokens par mot
Arabizi    16 tokens / 7 mots = 2.3 tokens par mot
Tounsi     14 tokens / 7 mots = 2.0 tokens par mot


In [4]:
import pandas as pd

# Recréer df_questions au cas où la cellule précédente n'a pas été exécutée dans cette session
if 'df_questions' not in globals():
    Q_CONTEXTE = [
        ("Français", "Quels sont les frais du Génie Informatique à l'IIT ?", "778", "non"),
        ("Français", "Quelles sont les conditions d'accès au Génie Mécanique ?", "préparatoire|prépa|MP|licence", "non"),
        ("Français", "Combien d'années dure la licence Business Administration de l'ISB ?", "3", "non"),
        ("Français", "Quels sont les frais d'inscription en plus des frais de scolarité à l'IIT ?", "800", "non"),
        ("Français", "Quels sont les frais de la licence Business Administration à l'ISB ?", "", "oui"),
        ("Anglais",  "What career opportunities are there after Industrial Engineering at IIT?", "logistic|quality|supply|production|consult", "non"),
        ("Anglais",  "Is the Business Administration bachelor at ISB taught in English?", "yes|english", "non"),
        ("Anglais",  "Which specializations are offered in Computer Engineering at IIT?", "GLID|ARSI|SDIA", "non"),
        ("Anglais",  "Which accreditation does the IIT Computer Engineering program have?", "ASIIN|Euro-Inf", "non"),
        ("Anglais",  "Does IIT offer scholarships for engineering students?", "", "oui"),
        ("Tounsi",   "قداش معاليم الهندسة المدنية في IIT ؟", "556|600", "non"),
        ("Tounsi",   "شنوة الخدمة اللي نجم نخدمها بعد الهندسة الصناعية ؟", "logisti|qualit|supply|لوجست|جودة|إنتاج", "non"),
        ("Tounsi",   "شنية الشروط باش نقرا هندسة الإعلامية ؟", "préparatoire|prépa|licence|إعداد|إجازة|ليسانس", "non"),
        ("Tounsi",   "قداش تزيد المعاليم بعد العام الأول ؟", "5 %|5%|10%", "non"),
        ("Tounsi",   "الهندسة الميكانيكية تتقرا بالفرنساوي ولا بالأنقليزي ؟", "", "oui"),
        ("Arabizi",  "9adeh soum genie info fel IIT ?", "778", "non"),
        ("Arabizi",  "chnouma les parcours elli fama fi genie informatique ?", "GLID|ARSI|SDIA", "non"),
        ("Arabizi",  "3aslema, billehi n7eb nes2lek 3al genie info, chnouma el conditions bech nod5ol ?", "préparatoire|prépa|licence", "non"),
        ("Arabizi",  "salam, ena 3andi bac math w n7eb na9ra fel IIT, chnouma el 5ayarat elli 3andi ?", "prépa|paratoire|licence", "non"),
        ("Arabizi",  "n7eb n5dem fel cybersécurité, ama parcours ynasbni fi genie info ?", "ARSI", "non"),
        ("Arabizi",  "famma frais d'inscription zeyda walla le ?", "800", "non"),
        ("Arabizi",  "9adeh soum genie electrique fel IIT ?", "", "oui"),
        ("Mélange",  "Bonjour, nheb n3ref les débouchés mta3 génie civil", "structure|construction|infrastructure|bureau", "non"),
        ("Mélange",  "Hi, n7eb na9ra business in English, famma programme fel ISB ?", "Business Administration|English|anglais", "non"),
        ("Mélange",  "Salut, genie des procédés yod5el fih el pétrole walla le ?", "pétrol|petrol|énergie|energie", "non"),
        ("Mélange",  "Hello, 9addech men 3am yodhher genie civil ?", "", "oui"),
    ]
    Q_SANS_CONTEXTE = [
        ("Arabizi",  "chnowa ma3neha 'barcha' bel fransa ?", "beaucoup", "non"),
        ("Arabizi",  "traduis en français : ena mech fehem chay", "comprends", "non"),
        ("Tounsi",   "شنوة معناها 'يعيشك' بالفرنسية ؟", "merci", "non"),
        ("Arabizi",  "kifech nqoul 'je suis fatigué' bel tounsi ?", "ta3ban|t3ban|تعبان|3yit|عييت", "non"),
        ("Arabizi",  "chnowa ma3neha 'taw' bel fransa ?", "maintenant|tout de suite|now", "non"),
        ("Tounsi",   "ترجم للفرنسية : نحب نقرا في الجامعة", "étudier|université|universite", "non"),
        ("Arabizi",  "9oulli bel arabizi : comment tu t'appelles ?", "esmek|ismek|chesmek|chnowa esmek|ism", "non"),
        ("Arabizi",  "3lech fel arabizi nekteb '3' w '7' ?", "ع|ح|lettre|letter|7arf|حرف", "non"),
        ("Mélange",  "rani stressé barcha 3al bac, 3andek nasi7a ?", "révis|organis|repos|sommeil|respir|plan|نوم", "non"),
        ("Arabizi",  "jawebni bel tounsi : chnowa a7sen wa9t bech na9ra fel nhar ?", "sbe7|sba7|matin|lil|صباح|ليل", "non"),
    ]
    lignes = [(f"C{i}", "contexte", *q) for i, q in enumerate(Q_CONTEXTE, 1)] + \
             [(f"D{i}", "sans_contexte", *q) for i, q in enumerate(Q_SANS_CONTEXTE, 1)]
    df_questions = pd.DataFrame(lignes, columns=["id", "mode", "langue", "question", "mots_attendus", "piege"])

# Calculer les statistiques de tokenisation
df_questions["tokens"] = df_questions["question"].apply(lambda q: len(tok.tokenize(q)))
df_questions["mots"] = df_questions["question"].apply(lambda q: len(q.split()))
res = df_questions.groupby("langue")[["tokens", "mots"]].sum()
res["tokens_par_mot"] = (res["tokens"] / res["mots"]).round(2)
res.sort_values("tokens_par_mot")

,tokens,mots,tokens_par_mot
langue,,,
Anglais,56,45,1.24
Français,88,55,1.60
Arabizi,259,126,2.06
Mélange,99,46,2.15
Tounsi,112,49,2.29
